In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install py-clob-client

In [ ]:

# ============================================================================
# IMPORTS
# ============================================================================

# Standard library imports
import json
import time
import datetime
import threading
from typing import Optional, Dict, List
from datetime import datetime, timezone, timedelta
import os

# Third-party imports
import requests
import numpy as np
import pandas as pd
from websocket import WebSocketApp
from py_clob_client.client import ClobClient

In [ ]:
host: str = "https://clob.polymarket.com"
key: str = "YOUR_KEY" #This is your Private Key. If using email login export from https://reveal.magic.link/polymarket otherwise export from your Web3 Application
chain_id: int = 137 #No need to adjust this
POLYMARKET_PROXY_ADDRESS: str = 'YOUR_WALLET' #This is the address you deposit/send USDC to to FUND your Polymarket account.
MARKET_CHANNEL = 'market'
USER_CHANNEL = 'user'

client = ClobClient(host, key=key, chain_id=chain_id, signature_type=1, funder=POLYMARKET_PROXY_ADDRESS)

print( client.derive_api_key() )

In [ ]:
def get_lob_bounds(client, token_id_yes, token_id_no, level=10):
    """
    Fetch bid_10 and ask_10 prices for both YES and NO tokens.

    Args:
        client: ClobClient instance
        token_id_yes: YES token CLOB ID
        token_id_no: NO token CLOB ID
        level: Which level to use (default: 10 for 10th level)

    Returns:
        Dict mapping asset_id to price bounds
    """
    # Fetch order books
    ob_yes = client.get_order_book(token_id_yes)
    ob_no = client.get_order_book(token_id_no)

    # Use negative indexing to get the correct level (matching your LOB snapshot logic)
    # -level means: -10 for level 10, -1 for level 1 (best bid/ask)

    # YES bounds (using negative indexing)
    if ob_yes.bids and len(ob_yes.bids) >= level:
        yes_bid_10 = float(ob_yes.bids[-level].price)
    else:
        yes_bid_10 = float(ob_yes.bids[-1].price) if ob_yes.bids else 0.0

    if ob_yes.asks and len(ob_yes.asks) >= level:
        yes_ask_10 = float(ob_yes.asks[-level].price)
    else:
        yes_ask_10 = float(ob_yes.asks[-1].price) if ob_yes.asks else 1.0

    # NO bounds (using negative indexing)
    if ob_no.bids and len(ob_no.bids) >= level:
        no_bid_10 = float(ob_no.bids[-level].price)
    else:
        no_bid_10 = float(ob_no.bids[-1].price) if ob_no.bids else 0.0

    if ob_no.asks and len(ob_no.asks) >= level:
        no_ask_10 = float(ob_no.asks[-level].price)
    else:
        no_ask_10 = float(ob_no.asks[-1].price) if ob_no.asks else 1.0

    print("Price Bounds Set:")
    print(f"YES: {yes_bid_10:.4f} - {yes_ask_10:.4f}")
    print(f"NO:  {no_bid_10:.4f} - {no_ask_10:.4f}\n")

    return {
        token_id_yes: {
            'min': yes_bid_10,
            'max': yes_ask_10,
            'side': 'YES'
        },
        token_id_no: {
            'min': no_bid_10,
            'max': no_ask_10,
            'side': 'NO'
        }
    }



In [ ]:
class PolymarketEventFetcher:
    """
    Fetch and analyze Polymarket recurring series events with LOB snapshots.

    Attributes:
        series_slug: The slug identifier for the Polymarket series
        base_url: Gamma API base URL
        data: DataFrame containing the active event data
        client: CLOB client for fetching order books
    """

    def __init__(self, series_slug: str, base_url: str = "https://gamma-api.polymarket.com"):
        """
        Initialize the fetcher with a series slug.

        Args:
            series_slug: The slug of the recurring series (e.g., 'daily-btc-price')
            base_url: Base URL for Gamma API (default: official endpoint)
        """
        self.series_slug = series_slug
        self.base_url = base_url
        self.data = None
        self.client = None


    def get_binance_settlement_price(self):
      """
      Get the most recent Binance BTC close price at 17:00 UTC


      - If current time is AFTER 17:00 UTC today → return today's 17:00 price
      - If current time is BEFORE 17:00 UTC today → return yesterday's 17:00 price


      Returns:
          float: Close price at most recent 17:00 UTC
      """



      now_utc = datetime.now(timezone.utc)


      # Determine which 17:00 UTC to fetch
      if now_utc.hour >= 17:
          # After 17:00 today → use today's 17:00
          target_date = now_utc
      else:
          # Before 17:00 today → use yesterday's 17:00
          target_date = now_utc - timedelta(days=1)


      # Set to exactly 17:00:00 UTC
      target_time = target_date.replace(hour=17, minute=0, second=0, microsecond=0)
      target_timestamp = int(target_time.timestamp() * 1000)


      # Fetch 1-minute kline at 17:00 UTC
      url = "https://api.binance.com/api/v3/klines"
      params = {
          "symbol": "BTCUSDT",
          "interval": "1m",
          "startTime": target_timestamp,
          "limit": 1
      }


      try:
          response = requests.get(url, params=params, timeout=5)
          data = response.json()


          if data and len(data) > 0:
              close_price = float(data[0][4])
              print(f"✅ Binance settlement price at {target_time.strftime('%Y-%m-%d %H:%M UTC')}: ${close_price:,.2f}")
              return close_price
          else:
              print("⚠️ No data returned from Binance")
              return None


      except Exception as e:
          print(f"❌ Error fetching Binance price: {e}")
          return None

    def get_binance_current_price(self):
      """
      Get the most recent Binance BTC close price


      Returns:
          float: Close price at most recent price of BTC
      """


      # Fetch 1-minute kline at 17:00 UTC
      url = "https://api.binance.com/api/v3/klines"
      params = {
          "symbol": "BTCUSDT",
          "interval": "1m",
          "limit": 1
      }


      try:
          response = requests.get(url, params=params, timeout=5)
          data = response.json()


          if data and len(data) > 0:
              close_price = float(data[0][4])
              return close_price
          else:
              print("⚠️ No data returned from Binance")
              return None


      except Exception as e:
          print(f"❌ Error fetching Binance price: {e}")
          return None


    def fetch_active_event(self) -> pd.DataFrame:
        """
        Fetch the currently active daily event for this series.

        Returns:
            DataFrame with active event data including CLOB token IDs

        Raises:
            ValueError: If no events found or no active events match criteria
            ConnectionError: If API request fails
        """
        try:
            # Fetch series data
            response = requests.get(
                f"{self.base_url}/series",
                params={'slug': self.series_slug},
                timeout=10
            )
            response.raise_for_status()
            series_data = response.json()

            # Validate response
            if not series_data or 'events' not in series_data[0]:
                raise ValueError(f"No events found for series: {self.series_slug}")

            # Parse events
            df = pd.DataFrame(series_data[0]['events'])

            if df.empty:
                raise ValueError("No events available")

            # Convert date columns efficiently
            df['creationDate'] = pd.to_datetime(df['creationDate'], format='ISO8601', utc=True)
            df['startDate'] = pd.to_datetime(df['startDate'], format='ISO8601', utc=True)
            df['endDate'] = pd.to_datetime(df['endDate'], format='ISO8601', utc=True)

            # Cache timestamp once for consistency and performance
            now = pd.Timestamp.now(tz='UTC')
            yesterday = now - pd.Timedelta(days=1)

            # Filter for active events (created in last 24 hours)
            df = df[
                df['active'] &
                ~df['closed'] &
                (df['creationDate'] >= yesterday) &
                (df['creationDate'] < now)
            ]

            if df.empty:
                raise ValueError("No active events match the criteria")

            # Select relevant columns
            columns = ['id', 'slug', 'title', 'volume', 'startDate', 'endDate']
            df = df[columns].reset_index(drop=True)

            # Fetch CLOB token IDs for first event
            df = self._add_clob_tokens(df)

            self.data = df
            return df

        except requests.RequestException as e:
            raise ConnectionError(f"Failed to fetch data from Polymarket: {e}")

    def _add_clob_tokens(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Add CLOB token IDs to the dataframe (private method).

        Args:
            df: DataFrame with event data

        Returns:
            DataFrame with added clob_token_id_YES and clob_token_id_NO columns
        """
        if df.empty:
            return df

        response = requests.get(
            f"{self.base_url}/events/slug/{df.loc[0, 'slug']}",
            timeout=10
        )
        response.raise_for_status()
        event_data = response.json()

        # The API returns clobTokenIds as a JSON string, parse it
        clob_tokens_str = event_data['markets'][0]['clobTokenIds']
        clob_tokens = json.loads(clob_tokens_str)

        df.loc[0, 'clob_token_id_YES'] = clob_tokens[0]
        df.loc[0, 'clob_token_id_NO'] = clob_tokens[1]

        return df

    def set_clob_client(self, client: ClobClient):
        """
        Set the CLOB client for fetching order books.

        Args:
            client: Initialized ClobClient instance
        """
        self.client = client

    def capture_lob_snapshot(self, levels: int = 10, settlement_price = None) -> Dict:
        """
        Capture a single limit order book snapshot for the active event.

        Args:
            levels: Number of order book levels to capture (default: 10)

        Returns:
            Dictionary with 'YES' and 'NO' DataFrames containing LOB data

        Raises:
            ValueError: If no active event data or CLOB client not set
        """

        now = datetime.now(timezone.utc)
        target = now.replace(hour=17, minute=0, second=0, microsecond=0)
        if now >= target:
          target += timedelta(days=1)

        now = now.timestamp()
        target = target.timestamp()

        if self.data is None or self.data.empty:
            raise ValueError("No active event data. Call fetch_active_event() first.")

        if self.client is None:
            raise ValueError("CLOB client not set. Call set_clob_client() first.")

        event = self.data.iloc[0]
        ts = datetime.now()

        # Fetch order books with error handling
        try:
            ob_yes = self.client.get_order_book(event['clob_token_id_YES'])
        except Exception as e:
            print(f"Warning: Failed to fetch YES order book: {e}")
            ob_yes = None

        try:
            ob_no = self.client.get_order_book(event['clob_token_id_NO'])
        except Exception as e:
            print(f"Warning: Failed to fetch NO order book: {e}")
            ob_no = None

        # Get volume with error handling
        vol = np.nan
        try:
            vol_response = requests.get(
                'https://data-api.polymarket.com/live-volume',
                params={'id': event['id']},
                timeout=10
            )
            vol_response.raise_for_status()
            vol = float(vol_response.json()[0]['total'])
        except Exception as e:
            print(f"Warning: Failed to fetch volume: {e}")

        # Build YES data dictionary with error handling
        yes_data = {}
        if ob_yes and ob_yes.bids and ob_yes.asks:
            try:
                for i in range(1, levels + 1):
                    # Check if level exists
                    if i <= len(ob_yes.bids) and i <= len(ob_yes.asks):
                        yes_data[f'bid_{i}'] = float(ob_yes.bids[-i].price)
                        yes_data[f'ask_{i}'] = float(ob_yes.asks[-i].price)
                        yes_data[f'bidsize_{i}'] = float(ob_yes.bids[-i].size)
                        yes_data[f'asksize_{i}'] = float(ob_yes.asks[-i].size)

                    else:
                        # Fill with NaN if level doesn't exist
                        yes_data[f'bid_{i}'] = np.nan
                        yes_data[f'ask_{i}'] = np.nan
                        yes_data[f'bidsize_{i}'] = np.nan
                        yes_data[f'asksize_{i}'] = np.nan

                # Calculate spread
                yes_data['spread'] = float(ob_yes.asks[-1].price) - float(ob_yes.bids[-1].price)
            except Exception as e:
                print(f"Warning: Error building YES data: {e}")
                # Fill with NaN
                for i in range(1, levels + 1):
                    yes_data[f'bid_{i}'] = np.nan
                    yes_data[f'ask_{i}'] = np.nan
                    yes_data[f'bidsize_{i}'] = np.nan
                    yes_data[f'asksize_{i}'] = np.nan
                yes_data['spread'] = np.nan
        else:
            # Order book is empty or None
            for i in range(1, levels + 1):
                yes_data[f'bid_{i}'] = np.nan
                yes_data[f'ask_{i}'] = np.nan
                yes_data[f'bidsize_{i}'] = np.nan
                yes_data[f'asksize_{i}'] = np.nan
            yes_data['spread'] = np.nan

        # Add metadata to YES
        yes_data['snapshot_timestamp'] = ts
        yes_data['total_volume_(yes_and_no)'] = vol
        yes_data['settlement_price'] = settlement_price
        yes_data['current_price'] = self.get_binance_current_price()
        yes_data['price_diff'] = yes_data['current_price']/yes_data['settlement_price'] - 1
        yes_data['time_to_expiry'] = target - now
        # Build NO data dictionary with error handling
        no_data = {}
        if ob_no and ob_no.bids and ob_no.asks:
            try:
                for i in range(1, levels + 1):
                    # Check if level exists
                    if i <= len(ob_no.bids) and i <= len(ob_no.asks):
                        no_data[f'bid_{i}'] = float(ob_no.bids[-i].price)
                        no_data[f'ask_{i}'] = float(ob_no.asks[-i].price)
                        no_data[f'bidsize_{i}'] = float(ob_no.bids[-i].size)
                        no_data[f'asksize_{i}'] = float(ob_no.asks[-i].size)
                    else:
                        # Fill with NaN if level doesn't exist
                        no_data[f'bid_{i}'] = np.nan
                        no_data[f'ask_{i}'] = np.nan
                        no_data[f'bidsize_{i}'] = np.nan
                        no_data[f'asksize_{i}'] = np.nan

                # Calculate spread
                no_data['spread'] = float(ob_no.asks[-1].price) - float(ob_no.bids[-1].price)
            except Exception as e:
                print(f"Warning: Error building NO data: {e}")
                # Fill with NaN
                for i in range(1, levels + 1):
                    no_data[f'bid_{i}'] = np.nan
                    no_data[f'ask_{i}'] = np.nan
                    no_data[f'bidsize_{i}'] = np.nan
                    no_data[f'asksize_{i}'] = np.nan
                no_data['spread'] = np.nan
        else:
            # Order book is empty or None
            for i in range(1, levels + 1):
                no_data[f'bid_{i}'] = np.nan
                no_data[f'ask_{i}'] = np.nan
                no_data[f'bidsize_{i}'] = np.nan
                no_data[f'asksize_{i}'] = np.nan
            no_data['spread'] = np.nan

        # Add metadata to NO
        no_data['snapshot_timestamp'] = ts
        no_data['total_volume_(yes_and_no)'] = vol
        no_data['settlement_price'] = settlement_price
        no_data['current_price'] = self.get_binance_current_price()
        no_data['price_diff'] = no_data['current_price']/no_data['settlement_price'] - 1
        no_data['time_to_expiry'] = target - now

        # Create DataFrames from dictionaries
        return {
            'YES': pd.DataFrame([yes_data]),
            'NO': pd.DataFrame([no_data]),
            'event_id': event['id'],
            'event_slug': event['slug']
        }

    def capture_multiple_lob_snapshots(
        self,
        num_snapshots: int = 5,
        interval_seconds: int = 3,
        levels: int = 10,
        verbose: bool = True
    ) -> Dict:
        """
        Capture multiple LOB snapshots over time and return concatenated DataFrames.

        Args:
            num_snapshots: Number of snapshots to capture (default: 5)
            interval_seconds: Seconds to wait between snapshots (default: 3)
            levels: Number of order book levels to capture (default: 10)
            verbose: Print progress messages (default: True)

        Returns:
            Dictionary with:
                - 'YES': DataFrame with all YES snapshots
                - 'NO': DataFrame with all NO snapshots
                - 'event_id': Event ID
                - 'event_slug': Event slug
        """
        if self.data is None or self.data.empty:
            raise ValueError("No active event data. Call fetch_active_event() first.")

        if self.client is None:
            raise ValueError("CLOB client not set. Call set_clob_client() first.")

        snapshots = []

        for i in range(num_snapshots):
            snapshot = self.capture_lob_snapshot(levels=levels)
            snapshots.append(snapshot)

            if verbose:
                ts = snapshot['YES']['snapshot_timestamp'].iloc[0]
                print(f"Snapshot {i+1}/{num_snapshots} captured at {ts}")

            # Sleep between snapshots (except after the last one)
            if i < num_snapshots - 1:
                time.sleep(interval_seconds)

        # Concatenate all snapshots efficiently
        all_yes = pd.concat([s['YES'] for s in snapshots], ignore_index=True)
        all_no = pd.concat([s['NO'] for s in snapshots], ignore_index=True)

        return {
            'YES': all_yes,
            'NO': all_no,
            'event_id': snapshots[0]['event_id'],
            'event_slug': snapshots[0]['event_slug']
        }

    def monitor_market_activity(
      self,
      duration_seconds: int = 300,
      snapshot_interval: int = 30,
      ws_interval: int = 5,
      lob_levels: int = 10,
      verbose: bool = True
  ) -> Dict:
      """
      Monitor market activity with periodic LOB snapshots and real-time WebSocket data.

      This unified method runs both LOB snapshot collection and WebSocket monitoring
      in parallel, providing comprehensive market data over an extended period.

      Args:
          duration_seconds: Total monitoring duration (default: 300s / 5 minutes)
          snapshot_interval: Seconds between LOB snapshots (default: 30s)
          ws_interval: Seconds for WebSocket data aggregation intervals (default: 5s)
          lob_levels: Number of order book levels to capture (default: 10)
          verbose: Print progress messages (default: True)

      Returns:
          Dict with:
              - 'lob_snapshots': Dict with 'YES' and 'NO' DataFrames of all LOB snapshots
              - 'price_changes': Dict with 'YES' and 'NO' price change counts
              - 'trades': Dict with 'YES' and 'NO' trade DataFrames
              - 'intervals': DataFrame with interval-by-interval tracking
              - 'summary': Dict with aggregate statistics
              - 'event_info': Dict with event metadata
      """
      if self.data is None or self.data.empty:
          raise ValueError("No active event data. Call fetch_active_event() first.")

      if self.client is None:
          raise ValueError("CLOB client not set. Call set_clob_client() first.")

      event = self.data.iloc[0]
      settlement_price = self.get_binance_settlement_price()

      print(f"\n{'='*60}")
      print(f"Starting Market Monitoring")
      print(f"{'='*60}")
      print(f"Event: {event['title']}")
      print(f"Duration: {duration_seconds}s ({duration_seconds/60:.1f} minutes)")
      print(f"Snapshot Interval: {snapshot_interval}s")
      print(f"WebSocket Interval: {ws_interval}s")
      print(f"LOB Levels: {lob_levels}")
      print(f"{'='*60}\n")

      # Get initial LOB bounds for WebSocket filtering
      token_bounds = get_lob_bounds(
          self.client,
          event['clob_token_id_YES'],
          event['clob_token_id_NO'],
          level=lob_levels
      )

      # Shared data structure for results
      results = {
          'lob_snapshots_yes': [],
          'lob_snapshots_no': [],
          'ws_monitor': None,
          'completed': False
      }

      # Thread for WebSocket monitoring
      def websocket_thread():
          url = "wss://ws-subscriptions-clob.polymarket.com"
          api_key = self.client.derive_api_key().api_key
          api_secret = self.client.derive_api_key().api_secret
          api_passphrase = self.client.derive_api_key().api_passphrase

          auth = {
              "apiKey": api_key,
              "secret": api_secret,
              "passphrase": api_passphrase
          }

          # ✅ UPDATED: Pass additional parameters for dynamic bound updates
          ws_monitor = WebSocketOrderBook(
              MARKET_CHANNEL,
              url,
              list(token_bounds.keys()),
              auth,
              None,
              verbose=False,
              token_bounds=token_bounds,
              duration_seconds=duration_seconds,
              interval_seconds=ws_interval,
              client=self.client,  # ✅ Pass client for bound updates
              token_id_yes=event['clob_token_id_YES'],  # ✅ Pass YES token ID
              token_id_no=event['clob_token_id_NO'],  # ✅ Pass NO token ID
              lob_level=lob_levels  # ✅ Pass LOB level
          )

          results['ws_monitor'] = ws_monitor
          ws_monitor.run()

      # Thread for LOB snapshots (NO CHANGES NEEDED)
      def snapshot_thread():
          start_time = time.time()
          snapshot_count = 0

          while time.time() - start_time < duration_seconds:
              try:
                  snapshot_count += 1
                  elapsed = time.time() - start_time

                  if verbose:
                      print(f"[{elapsed:.0f}s] Capturing LOB snapshot #{snapshot_count}...")

                  snapshot = self.capture_lob_snapshot(levels=lob_levels, settlement_price = settlement_price)

                  # Add elapsed time to snapshot
                  snapshot['YES']['elapsed_seconds'] = elapsed
                  snapshot['NO']['elapsed_seconds'] = elapsed

                  results['lob_snapshots_yes'].append(snapshot['YES'])
                  results['lob_snapshots_no'].append(snapshot['NO'])

                  # Wait for next snapshot or end
                  remaining = duration_seconds - (time.time() - start_time)
                  if remaining > 0:
                      sleep_time = min(snapshot_interval, remaining)
                      time.sleep(sleep_time)
                  else:
                      break

              except Exception as e:
                  print(f"Error capturing snapshot: {e}")
                  time.sleep(snapshot_interval)

          results['completed'] = True
          if verbose:
              print(f"\n[{duration_seconds}s] LOB snapshot collection completed.")

      # Start both threads
      ws_thread = threading.Thread(target=websocket_thread)
      snap_thread = threading.Thread(target=snapshot_thread)

      ws_thread.start()
      snap_thread.start()

      # Wait for both to complete
      ws_thread.join()
      snap_thread.join()

      # Compile results
      ws_monitor = results['ws_monitor']

      # Concatenate all LOB snapshots
      lob_yes = pd.concat(results['lob_snapshots_yes'], ignore_index=True) if results['lob_snapshots_yes'] else pd.DataFrame()
      lob_no = pd.concat(results['lob_snapshots_no'], ignore_index=True) if results['lob_snapshots_no'] else pd.DataFrame()

      # Get trade DataFrames from WebSocket
      trade_dfs = ws_monitor.get_trade_dataframes()

      # Get interval DataFrame
      interval_df = ws_monitor.get_interval_dataframe()

      # Calculate summary statistics
      summary = self._calculate_summary(
          lob_yes, lob_no,
          ws_monitor.total_price_change_counts,
          trade_dfs,
          duration_seconds
      )

      print(f"\n{'='*60}")
      print(f"Monitoring Complete - Summary")
      print(f"{'='*60}")
      print(f"Duration: {duration_seconds}s")
      print(f"LOB Snapshots: {len(lob_yes)}")
      print(f"WebSocket Intervals: {len(interval_df)}")
      print(f"\nPrice Changes:")
      print(f"  YES: {ws_monitor.total_price_change_counts['YES']}")
      print(f"  NO:  {ws_monitor.total_price_change_counts['NO']}")
      print(f"\nTrades:")
      print(f"  YES: {len(trade_dfs['YES'])} trades")
      print(f"  NO:  {len(trade_dfs['NO'])} trades")
      if not trade_dfs['YES'].empty:
          print(f"  YES Volume: ${trade_dfs['YES']['size'].sum():.2f}")
      if not trade_dfs['NO'].empty:
          print(f"  NO Volume: ${trade_dfs['NO']['size'].sum():.2f}")
      print(f"{'='*60}\n")

      return {
          'lob_snapshots': {
              'YES': lob_yes,
              'NO': lob_no
          },
          'price_changes': ws_monitor.total_price_change_counts,
          'trades': trade_dfs,
          'intervals': interval_df,
          'summary': summary,
          'event_info': {
              'id': event['id'],
              'slug': event['slug'],
              'title': event['title']
          }
      }
    def _calculate_summary(self, lob_yes, lob_no, price_changes, trade_dfs, duration):
        """
        Calculate aggregate statistics from monitoring data.

        Args:
            lob_yes: YES LOB snapshots DataFrame
            lob_no: NO LOB snapshots DataFrame
            price_changes: Dict with price change counts
            trade_dfs: Dict with trade DataFrames
            duration: Monitoring duration in seconds

        Returns:
            Dict with summary statistics
        """
        summary = {
            'duration_seconds': duration,
            'lob_snapshot_count': len(lob_yes),
            'price_changes_total': sum(price_changes.values()),
            'trades_total': len(trade_dfs['YES']) + len(trade_dfs['NO'])
        }

        # LOB statistics
        if not lob_yes.empty:
            summary['yes_avg_spread'] = lob_yes['spread'].mean()
            summary['yes_min_spread'] = lob_yes['spread'].min()
            summary['yes_max_spread'] = lob_yes['spread'].max()
            summary['yes_spread_volatility'] = lob_yes['spread'].std()

        if not lob_no.empty:
            summary['no_avg_spread'] = lob_no['spread'].mean()
            summary['no_min_spread'] = lob_no['spread'].min()
            summary['no_max_spread'] = lob_no['spread'].max()
            summary['no_spread_volatility'] = lob_no['spread'].std()

        # Trade statistics
        if not trade_dfs['YES'].empty:
            summary['yes_trade_volume'] = trade_dfs['YES']['size'].sum()
            summary['yes_vwap'] = (trade_dfs['YES']['price'] * trade_dfs['YES']['size']).sum() / trade_dfs['YES']['size'].sum()
            summary['yes_buy_volume'] = trade_dfs['YES'][trade_dfs['YES']['side'] == 'BUY']['size'].sum()
            summary['yes_sell_volume'] = trade_dfs['YES'][trade_dfs['YES']['side'] == 'SELL']['size'].sum()

        if not trade_dfs['NO'].empty:
            summary['no_trade_volume'] = trade_dfs['NO']['size'].sum()
            summary['no_vwap'] = (trade_dfs['NO']['price'] * trade_dfs['NO']['size']).sum() / trade_dfs['NO']['size'].sum()
            summary['no_buy_volume'] = trade_dfs['NO'][trade_dfs['NO']['side'] == 'BUY']['size'].sum()
            summary['no_sell_volume'] = trade_dfs['NO'][trade_dfs['NO']['side'] == 'SELL']['size'].sum()

        return summary

    def run_continuous_cycle_seconds(
        self,
        total_seconds: int,          # ✅ total time budget in seconds
        snapshot_interval: int = 30,
        ws_interval: int = 30,
        lob_levels: int = 10,
        output_dir: str = 'data',
        cutoff_hour_utc: int = 17,
        verbose = False
    ):
        """
        Run monitoring continuously for a total number of seconds.
        Automatically handles the 17:00 UTC transition to new events.
        """
        from datetime import datetime, timezone, timedelta
        import time

        print(f"\n🚀 Starting Continuous Monitoring Cycle for {total_seconds} seconds")
        print(f"🔄 Event Reset Time: {cutoff_hour_utc}:00 UTC")

        start_time = datetime.now(timezone.utc)

        while True:
            elapsed = (datetime.now(timezone.utc) - start_time).total_seconds()
            remaining_budget = total_seconds - elapsed
            if remaining_budget <= 0:
                print("\n⏰ Total time budget reached. Stopping continuous cycle.")
                break

            # --- PHASE 1: FETCH ACTIVE EVENT ---
            print("\n🔍 Fetching active event...")
            while True:
                try:
                    self.fetch_active_event()
                    event = self.data.iloc[0]
                    print(f"✅ Active Event Found: {event['slug']}")
                    break
                except Exception as e:
                    print(f"⏳ Waiting for new event to appear... ({e})")
                    time.sleep(10)

            # --- PHASE 2: COMPUTE HOW LONG TO RUN THIS SESSION ---
            now = datetime.now(timezone.utc)
            target = now.replace(hour=cutoff_hour_utc, minute=0, second=0, microsecond=0)
            if now >= target:
                target += timedelta(days=1)

            # time until next 17:00 UTC
            seconds_to_cutoff = (target - now).total_seconds() - 10  # small buffer
            # respect global remaining budget
            duration_seconds = int(min(seconds_to_cutoff, remaining_budget))

            if duration_seconds < 60:
                print("⏳ Too close to cutoff or budget end. Waiting a bit and retrying...")
                time.sleep(30)
                continue

            print(f"⏱️ This session duration: {duration_seconds}s "
                  f"({duration_seconds/3600:.2f} hours); "
                  f"budget remaining after: {(remaining_budget-duration_seconds)/3600:.2f} hours")

            # --- PHASE 3: RUN ONE MONITORING SESSION ---
            try:
                results = self.monitor_market_activity(
                    duration_seconds=duration_seconds,
                    snapshot_interval=snapshot_interval,
                    ws_interval=ws_interval,
                    lob_levels=lob_levels,
                    verbose=verbose
                )

                # --- PHASE 4: SAVE DATA ---
                self.export_monitoring_results(results, output_dir=output_dir)

            except Exception as e:
                print(f"❌ Error during monitoring session: {e}")

            # --- PHASE 5: SMALL TRANSITION PAUSE ---
            print("\n🔄 Session finished. Short transition pause...")
            time.sleep(60)
            self.data = None


    def export_monitoring_results(self, results: Dict, output_dir: str = 'data'):
        """
        Export all monitoring results to CSV files.

        Args:
            results: Results dict from monitor_market_activity()
            output_dir: Directory to save files (default: 'data')
        """
        import os

        # Create output directory
        os.makedirs(output_dir, exist_ok=True)

        # Generate timestamp for filenames
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        event_slug = results['event_info']['slug']

        # Export LOB snapshots
        lob_yes = results['lob_snapshots']['YES']
        lob_no = results['lob_snapshots']['NO']

        if not lob_yes.empty:
            lob_yes.to_csv(f"{output_dir}/{event_slug}_lob_yes_{timestamp}.csv", index=False)
            print(f"✓ Saved YES LOB snapshots ({len(lob_yes)} rows)")

        if not lob_no.empty:
            lob_no.to_csv(f"{output_dir}/{event_slug}_lob_no_{timestamp}.csv", index=False)
            print(f"✓ Saved NO LOB snapshots ({len(lob_no)} rows)")

        # Export trades
        trades_yes = results['trades']['YES']
        trades_no = results['trades']['NO']

        if not trades_yes.empty:
            trades_yes.to_csv(f"{output_dir}/{event_slug}_trades_yes_{timestamp}.csv", index=False)
            print(f"✓ Saved YES trades ({len(trades_yes)} rows)")

        if not trades_no.empty:
            trades_no.to_csv(f"{output_dir}/{event_slug}_trades_no_{timestamp}.csv", index=False)
            print(f"✓ Saved NO trades ({len(trades_no)} rows)")

        # Export interval data
        intervals_df = results['intervals']
        if not intervals_df.empty:
            intervals_df.to_csv(f"{output_dir}/{event_slug}_intervals_{timestamp}.csv", index=False)
            print(f"✓ Saved interval data ({len(intervals_df)} intervals)")

        # Export summary
        summary_df = pd.DataFrame([results['summary']])
        summary_df.to_csv(f"{output_dir}/{event_slug}_summary_{timestamp}.csv", index=False)
        print(f"✓ Saved summary statistics")

        print(f"\n✓ All data exported to '{output_dir}/' directory")


In [ ]:
class WebSocketOrderBook:
    def __init__(self, channel_type, url, data, auth, message_callback, verbose,
                 token_bounds, duration_seconds=None, interval_seconds=5,
                 client=None, token_id_yes=None, token_id_no=None, lob_level=10):
        """
        Initialize WebSocket with token-specific price bounds and optional duration.

        Args:
            channel_type: 'market' or 'user' channel
            url: WebSocket URL
            data: Asset IDs or market data
            auth: Authentication credentials
            message_callback: Optional callback function
            verbose: Print detailed messages
            token_bounds: Dict mapping asset_id to {'min': bid_10, 'max': ask_10, 'side': 'YES'/'NO'}
            duration_seconds: How long to run (None = forever)
            interval_seconds: Interval duration for aggregating counts (default: 5)
            client: ClobClient instance for updating bounds
            token_id_yes: YES token ID for bound updates
            token_id_no: NO token ID for bound updates
            lob_level: LOB level to use for bounds (default: 10)
        """
        self.channel_type = channel_type
        self.url = url
        self.data = data
        self.auth = auth
        self.message_callback = message_callback
        self.verbose = verbose
        self.token_bounds = token_bounds
        self.duration_seconds = duration_seconds
        self.interval_seconds = interval_seconds
        self.start_time = None
        self.stop_flag = False

        # For dynamic bound updates
        self.client = client
        self.token_id_yes = token_id_yes
        self.token_id_no = token_id_no
        self.lob_level = lob_level

        # Reconnection settings
        self.max_reconnect_attempts = 1000
        self.reconnect_delay = 5  # seconds
        self.connection_lost = False
        self.reconnect_count = 0

        # Current interval counters (reset every interval)
        self.current_interval_price_changes = {
            'YES': 0,
            'NO': 0
        }

        self.current_interval_trades = {
            'YES': 0,
            'NO': 0
        }

        # Interval history storage
        self.interval_history = []

        # Trade data for YES and NO (full history)
        self.trades = {
            'YES': [],
            'NO': []
        }

        # Overall counters (for summary)
        self.total_price_change_counts = {
            'YES': 0,
            'NO': 0
        }

        furl = url + "/ws/" + channel_type
        self.ws = WebSocketApp(
            furl,
            on_message=self.on_message,
            on_error=self.on_error,
            on_close=self.on_close,
            on_open=self.on_open,
        )
        self.orderbooks = {}

        # Interval tracking thread
        self.interval_thread = None

        # Last message timestamp for connection health check
        self.last_message_time = time.time()

    def update_bounds(self):
        """Update LOB bounds dynamically."""
        if not self.client or not self.token_id_yes or not self.token_id_no:
            return

        try:
            # Fetch fresh order books
            ob_yes = self.client.get_order_book(self.token_id_yes)
            ob_no = self.client.get_order_book(self.token_id_no)

            # YES bounds (using negative indexing)
            if ob_yes.bids and len(ob_yes.bids) >= self.lob_level:
                yes_bid = float(ob_yes.bids[-self.lob_level].price)
            else:
                yes_bid = float(ob_yes.bids[-1].price) if ob_yes.bids else 0.0

            if ob_yes.asks and len(ob_yes.asks) >= self.lob_level:
                yes_ask = float(ob_yes.asks[-self.lob_level].price)
            else:
                yes_ask = float(ob_yes.asks[-1].price) if ob_yes.asks else 1.0

            # NO bounds (using negative indexing)
            if ob_no.bids and len(ob_no.bids) >= self.lob_level:
                no_bid = float(ob_no.bids[-self.lob_level].price)
            else:
                no_bid = float(ob_no.bids[-1].price) if ob_no.bids else 0.0

            if ob_no.asks and len(ob_no.asks) >= self.lob_level:
                no_ask = float(ob_no.asks[-self.lob_level].price)
            else:
                no_ask = float(ob_no.asks[-1].price) if ob_no.asks else 1.0

            # Update bounds
            self.token_bounds[self.token_id_yes] = {
                'min': yes_bid,
                'max': yes_ask,
                'side': 'YES'
            }
            self.token_bounds[self.token_id_no] = {
                'min': no_bid,
                'max': no_ask,
                'side': 'NO'
            }

            if self.verbose:
                print(f"\n[Bounds Updated]")
                print(f"YES: {yes_bid:.4f} - {yes_ask:.4f}")
                print(f"NO:  {no_bid:.4f} - {no_ask:.4f}")

        except Exception as e:
            print(f"Warning: Failed to update bounds: {e}")

    def on_message(self, ws, message):
        """Handle incoming WebSocket messages."""
        # Update last message timestamp
        self.last_message_time = time.time()

        # Skip ping/pong messages
        if message == "PONG":
            return

        try:
            data = json.loads(message)
            event_type = data.get('event_type')

            # Handle price_change events
            if event_type == 'price_change':
                price_changes = data.get('price_changes', [])

                if not isinstance(price_changes, list):
                    return

                for change in price_changes:
                    if not isinstance(change, dict):
                        continue

                    asset_id = change.get('asset_id')
                    price = float(change.get('price', 0))

                    if asset_id in self.token_bounds:
                        bounds = self.token_bounds[asset_id]
                        min_price = bounds['min']
                        max_price = bounds['max']
                        side = bounds['side']

                        if min_price <= price <= max_price:
                            self.current_interval_price_changes[side] += 1
                            self.total_price_change_counts[side] += 1

                            if self.verbose:
                                print(f"\n[{side}] Price change (Interval: {self.current_interval_price_changes[side]})")
                                print(f"Asset: {asset_id[:20]}...")
                                print(f"Price: {price} (range: {min_price} - {max_price})")
                                print(f"Size: {change.get('size')}")

            # Handle last_trade_price events
            elif event_type == 'last_trade_price':
                asset_id = data.get('asset_id')

                if asset_id in self.token_bounds:
                    side = self.token_bounds[asset_id]['side']

                    self.current_interval_trades[side] += 1

                    trade_info = {
                        'price': float(data.get('price', 0)),
                        'size': float(data.get('size', 0)),
                        'side': data.get('side'),
                        'timestamp': data.get('timestamp'),
                        'fee_rate_bps': data.get('fee_rate_bps', '0')
                    }

                    self.trades[side].append(trade_info)

                    if self.verbose:
                        print(f"\n[{side}] Trade (Interval: {self.current_interval_trades[side]})")
                        print(f"Price: {trade_info['price']}")
                        print(f"Size: {trade_info['size']}")

        except json.JSONDecodeError:
            pass
        except Exception as e:
            if self.verbose:
                print(f"Error processing message: {e}")

    def interval_tracker(self):
        """Track intervals and save data every interval_seconds."""
        interval_number = 0

        while not self.stop_flag:
            time.sleep(self.interval_seconds)

            if self.stop_flag:
                break

            interval_number += 1
            timestamp = datetime.now()

            # Check connection health (no messages for 60 seconds = dead connection)
            if time.time() - self.last_message_time > 60:
                print(f"\n⚠️ Warning: No WebSocket messages received for 60 seconds")
                print(f"   Connection may be dead. Attempting reconnect...")
                self.connection_lost = True
                # Trigger reconnection by closing WebSocket
                try:
                    self.ws.close()
                except:
                    pass

            # Get current bounds before updating
            yes_bounds = self.token_bounds.get(self.token_id_yes, {})
            no_bounds = self.token_bounds.get(self.token_id_no, {})

            # Save current interval data
            interval_data = {
                'interval_number': interval_number,
                'timestamp': timestamp,
                'yes_price_changes': self.current_interval_price_changes['YES'],
                'no_price_changes': self.current_interval_price_changes['NO'],
                'total_price_changes': self.current_interval_price_changes['YES'] + self.current_interval_price_changes['NO'],
                'yes_trade_count': self.current_interval_trades['YES'],
                'no_trade_count': self.current_interval_trades['NO'],
                'total_trade_count': self.current_interval_trades['YES'] + self.current_interval_trades['NO'],
                'yes_bound_min': yes_bounds.get('min', np.nan),
                'yes_bound_max': yes_bounds.get('max', np.nan),
                'no_bound_min': no_bounds.get('min', np.nan),
                'no_bound_max': no_bounds.get('max', np.nan),
                'reconnect_count': self.reconnect_count
            }

            self.interval_history.append(interval_data)

            if self.verbose:
                print(f"\n{'='*60}")
                print(f"Interval {interval_number} Complete")
                print(f"YES Price Changes: {interval_data['yes_price_changes']}")
                print(f"NO Price Changes: {interval_data['no_price_changes']}")
                print(f"YES Trades: {interval_data['yes_trade_count']}")
                print(f"NO Trades: {interval_data['no_trade_count']}")
                print(f"{'='*60}\n")

            # Reset interval counters
            self.current_interval_price_changes = {'YES': 0, 'NO': 0}
            self.current_interval_trades = {'YES': 0, 'NO': 0}

            # Update bounds for next interval
            self.update_bounds()

    def on_error(self, ws, error):
        """Handle WebSocket errors."""
        print(f"❌ WebSocket Error: {error}")
        # Don't set stop_flag - allow reconnection

    def on_close(self, ws, close_status_code, close_msg):
        """Handle WebSocket connection close."""
        print(f"\n⚠️ WebSocket connection closed: {close_msg}")

        # Attempt reconnection if not intentionally stopped
        if not self.stop_flag and self.reconnect_count < self.max_reconnect_attempts:
            self.reconnect_count += 1
            print(f"🔄 Attempting reconnection #{self.reconnect_count}/{self.max_reconnect_attempts}...")
            print(f"   Waiting {self.reconnect_delay} seconds...")
            time.sleep(self.reconnect_delay)

            # Recreate WebSocket
            furl = self.url + "/ws/" + self.channel_type
            self.ws = WebSocketApp(
                furl,
                on_message=self.on_message,
                on_error=self.on_error,
                on_close=self.on_close,
                on_open=self.on_open,
            )

            # Run forever again
            self.ws.run_forever()
        else:
            # Final close - show summary
            print("\n" + "="*60)
            print("WebSocket Closed - Summary")
            print("="*60)
            print(f"\nTotal Reconnections: {self.reconnect_count}")
            print(f"Total Intervals: {len(self.interval_history)}")
            print("\nTotal Price Changes:")
            print(f"  YES: {self.total_price_change_counts['YES']}")
            print(f"  NO:  {self.total_price_change_counts['NO']}")
            print(f"  Total: {sum(self.total_price_change_counts.values())}")

            print("\nTotal Trades:")
            print(f"  YES: {len(self.trades['YES'])} trades")
            print(f"  NO:  {len(self.trades['NO'])} trades")
            print(f"  Total: {len(self.trades['YES']) + len(self.trades['NO'])} trades")

            if self.trades['YES']:
                yes_volume = sum(t['size'] for t in self.trades['YES'])
                yes_avg_price = sum(t['price'] * t['size'] for t in self.trades['YES']) / yes_volume if yes_volume > 0 else 0
                print(f"\n  YES Volume: {yes_volume:.2f}")
                print(f"  YES Avg Price: {yes_avg_price:.4f}")

            if self.trades['NO']:
                no_volume = sum(t['size'] for t in self.trades['NO'])
                no_avg_price = sum(t['price'] * t['size'] for t in self.trades['NO']) / no_volume if no_volume > 0 else 0
                print(f"\n  NO Volume: {no_volume:.2f}")
                print(f"  NO Avg Price: {no_avg_price:.4f}")

            print("="*60)

    def on_open(self, ws):
        """Handle WebSocket connection open."""
        if self.start_time is None:
            self.start_time = time.time()

        # Log reconnection
        if self.reconnect_count > 0:
            print(f"✅ WebSocket reconnected successfully (attempt #{self.reconnect_count})")

        # Reset connection health
        self.last_message_time = time.time()
        self.connection_lost = False

        if self.channel_type == MARKET_CHANNEL:
            ws.send(json.dumps({
                "assets_ids": list(self.token_bounds.keys()),
                "type": MARKET_CHANNEL
            }))

            ws.send(json.dumps({
                "assets_ids": list(self.token_bounds.keys()),
                "operation": "subscribe"
            }))

        elif self.channel_type == USER_CHANNEL and self.auth:
            ws.send(json.dumps({
                "markets": self.data,
                "type": USER_CHANNEL,
                "auth": self.auth
            }))

        # Start ping thread (only once)
        if self.reconnect_count == 0:
            ping_thread = threading.Thread(target=self.ping, args=(ws,))
            ping_thread.daemon = True
            ping_thread.start()

            # Start interval tracking thread (only once)
            self.interval_thread = threading.Thread(target=self.interval_tracker)
            self.interval_thread.daemon = True
            self.interval_thread.start()

            # Start timer thread if duration specified (only once)
            if self.duration_seconds:
                timer_thread = threading.Thread(target=self.timer, args=(ws,))
                timer_thread.daemon = True
                timer_thread.start()

    def timer(self, ws):
        """Stop WebSocket after specified duration."""
        time.sleep(self.duration_seconds)
        print(f"\n\n⏰ Time limit reached ({self.duration_seconds}s). Closing connection...")
        self.stop_flag = True
        ws.close()

    def ping(self, ws):
        """Send periodic ping messages to keep connection alive."""
        while not self.stop_flag:
            try:
                ws.send("PING")
                time.sleep(10)
            except:
                break

    def subscribe_to_tokens_ids(self, assets_ids):
        """Subscribe to additional token IDs."""
        if self.channel_type == MARKET_CHANNEL:
            self.ws.send(json.dumps({"assets_ids": assets_ids, "operation": "subscribe"}))

    def unsubscribe_to_tokens_ids(self, assets_ids):
        """Unsubscribe from token IDs."""
        if self.channel_type == MARKET_CHANNEL:
            self.ws.send(json.dumps({"assets_ids": assets_ids, "operation": "unsubscribe"}))

    def run(self):
        """Run WebSocket connection and return results."""
        self.ws.run_forever()
        return {
            'price_changes': self.total_price_change_counts,
            'trades': self.trades,
            'intervals': self.interval_history
        }

    def get_trade_dataframes(self):
        """Convert trade data to pandas DataFrames."""
        yes_df = pd.DataFrame(self.trades['YES']) if self.trades['YES'] else pd.DataFrame()
        no_df = pd.DataFrame(self.trades['NO']) if self.trades['NO'] else pd.DataFrame()

        if not yes_df.empty:
            yes_df['timestamp'] = pd.to_datetime(yes_df['timestamp'].astype(float), unit='ms', utc=True)
        if not no_df.empty:
            no_df['timestamp'] = pd.to_datetime(no_df['timestamp'].astype(float), unit='ms', utc=True)

        return {'YES': yes_df, 'NO': no_df}

    def get_interval_dataframe(self):
        """Convert interval history to pandas DataFrame."""
        return pd.DataFrame(self.interval_history)


In [ ]:
# 1. Initialize
fetcher = PolymarketEventFetcher('btc-up-or-down-daily')

# 2. Setup CLOB client
fetcher.set_clob_client(client)

# 3. Monitor with interval tracking


seconds_budget = int(5400)

fetcher.run_continuous_cycle_seconds(
    total_seconds=seconds_budget,
    snapshot_interval=30,
    ws_interval=30,
    lob_levels=10,
    output_dir='/content/drive/MyDrive/PolyMarket_LOB_Trial_7',
    cutoff_hour_utc=17,
    verbose = True
)



🚀 Starting Continuous Monitoring Cycle for 5400 seconds
🔄 Event Reset Time: 17:00 UTC

🔍 Fetching active event...
✅ Active Event Found: bitcoin-up-or-down-on-february-11
⏱️ This session duration: 3180s (0.88 hours); budget remaining after: 0.62 hours
✅ Binance settlement price at 2026-02-10 17:00 UTC: $69,432.00

Starting Market Monitoring
Event: Bitcoin Up or Down on February 11?
Duration: 3180s (53.0 minutes)
Snapshot Interval: 30s
WebSocket Interval: 30s
LOB Levels: 10

Price Bounds Set:
YES: 0.0020 - 0.0260
NO:  0.9740 - 0.9980

[0s] Capturing LOB snapshot #1...
[31s] Capturing LOB snapshot #2...
[62s] Capturing LOB snapshot #3...
[93s] Capturing LOB snapshot #4...
[123s] Capturing LOB snapshot #5...
[154s] Capturing LOB snapshot #6...
[185s] Capturing LOB snapshot #7...
[216s] Capturing LOB snapshot #8...
[247s] Capturing LOB snapshot #9...
[278s] Capturing LOB snapshot #10...
[309s] Capturing LOB snapshot #11...
[340s] Capturing LOB snapshot #12...
[370s] Capturing LOB snapshot 